In [1]:
import networkx as nx

%load_ext autoreload
%autoreload 2

from utgen.raggraph.utils import get_node_context

In [2]:
# Load the GraphML file
g = nx.read_graphml("../data/repo_graph.graphml")

In [3]:
print("Nodes:", g.number_of_nodes())
print("Edges:", g.number_of_edges())
print("Is directed:", g.is_directed())
print("Graph type:", type(g))

Nodes: 40
Edges: 44
Is directed: True
Graph type: <class 'networkx.classes.digraph.DiGraph'>


In [5]:
node_id = "utgen/raggraph/walker.py::function::iter_python_files"

context = get_node_context(g=g, node_id=node_id)
print(context)

### NODE INFO ###
file::type::name -> utgen/raggraph/walker.py::function::iter_python_files

source:
def iter_python_files(path, skip_init=True):
    """
    Iterate over .py files inside `path`, optionally skipping __init__.py.
    """
    for root, _, files in os.walk(path):
        for f in files:
            if f.endswith(".py"):
                if skip_init and f == "__init__.py":
                    continue
                yield Path(root, f).as_posix()

### INCOMING EDGES ###
utgen/raggraph/walker.py -[defines]-> utgen/raggraph/walker.py::function::iter_python_files
utgen/raggraph/walker.py::function::build_graph_from_directory -[calls]-> utgen/raggraph/walker.py::function::iter_python_files

### NEIGHBOR NODE DETAILS ###
--- Neighbor Node ---
file::type::name -> utgen/raggraph/walker.py::function::build_graph_from_directory
signature: def build_graph_from_directory(code_path: str, skip_init: bool, save_graph: bool, save_path: str)
docstring:
None



In [7]:
node_id = "utgen/raggraph/utils.py::function::normalize_signature"

context = get_node_context(g=g, node_id=node_id)
print(context)

### NODE INFO ###
file::type::name -> utgen/raggraph/utils.py::function::normalize_signature

source:
def normalize_signature(node):
    """Return a Python-like signature including annotations and return type."""
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return ""

    def unparse(x):
        try:
            return ast.unparse(x)
        except Exception:
            return ""

    parts = []

    # Pos-only args (Python 3.8+)
    for a in getattr(node.args, "posonlyargs", []):
        s = a.arg
        if a.annotation:
            s += f": {unparse(a.annotation)}"
        parts.append(s)
    if getattr(node.args, "posonlyargs", []):
        parts.append("/")

    # Regular args
    for a in node.args.args:
        s = a.arg
        if a.annotation:
            s += f": {unparse(a.annotation)}"
        parts.append(s)

    # Vararg
    if node.args.vararg:
        s = "*" + node.args.vararg.arg
        if node.args.vararg.annotation:
            s +